In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

customers = pd.read_csv("C:/Users/lenovo/Desktop/FinRisk-X/data/raw/customers.csv")
transactions = pd.read_csv("C:/Users/lenovo/Desktop/FinRisk-X/data/raw/transactions.csv")

transactions["timestamp"] = pd.to_datetime(
    transactions["timestamp"]
)

transactions = transactions.sort_values(
    ["customer_id", "timestamp"]
).reset_index(drop=True)

print("Customers:", customers.shape)
print("Transactions:", transactions.shape)

transactions.head()

Customers: (10000, 13)
Transactions: (200000, 12)


,transaction_id,customer_id,timestamp,transaction_type,merchant_id,merchant_category,device_id,is_fraud,amount,hour,day_of_week,is_weekend
0,1042065,100001,2025-03-29 21:37:35,Card,11346,Financial,20714,0,13351.43,21,5,1
1,968859,100001,2025-05-04 10:28:58,Card,10253,Healthcare,20821,0,4474.74,10,6,1
2,1025968,100001,2025-05-07 06:23:39,UPI,11753,Electronics,22961,0,11412.17,6,2,0
3,940082,100001,2025-05-08 15:05:13,Card,11663,Travel,22245,0,2012.79,15,3,0
4,915192,100001,2025-05-16 13:25:24,UPI,10372,Retail,22056,0,398.77,13,4,0


In [5]:
transactions["month"] = (
    transactions["timestamp"].dt.to_period("M")
)

monthly = (
    transactions
    .groupby(["customer_id", "month"])
    .agg(
        transaction_count=("transaction_id", "count"),
        total_spending=("amount", "sum"),
        average_transaction=("amount", "mean"),
        spending_std=("amount", "std"),
        maximum_transaction=("amount", "max"),
        fraud_count=("is_fraud", "sum")
    )
    .reset_index()
)

monthly.head()

,customer_id,month,transaction_count,total_spending,average_transaction,spending_std,maximum_transaction,fraud_count
0,100001,2025-03,1,13351.43,13351.430000,NaN,13351.43,0
1,100001,2025-05,4,18298.47,4574.617500,4856.705406,11412.17,0
2,100001,2025-06,3,3416.93,1138.976667,870.263635,1941.71,0
3,100001,2025-07,2,9934.52,4967.260000,1834.956239,6264.77,0
4,100001,2025-08,3,10318.99,3439.663333,4409.416534,8487.35,0


In [6]:
monthly["fraud_rate"] = (
    monthly["fraud_count"]
    / monthly["transaction_count"]
)

monthly["fraud_rate"] = monthly["fraud_rate"].fillna(0)

monthly.head()

,customer_id,month,transaction_count,total_spending,average_transaction,spending_std,maximum_transaction,fraud_count,fraud_rate
0,100001,2025-03,1,13351.43,13351.430000,NaN,13351.43,0,0.0
1,100001,2025-05,4,18298.47,4574.617500,4856.705406,11412.17,0,0.0
2,100001,2025-06,3,3416.93,1138.976667,870.263635,1941.71,0,0.0
3,100001,2025-07,2,9934.52,4967.260000,1834.956239,6264.77,0,0.0
4,100001,2025-08,3,10318.99,3439.663333,4409.416534,8487.35,0,0.0


In [7]:
monthly = monthly.sort_values(
    ["customer_id", "month"]
)

monthly["spending_change"] = (
    monthly
    .groupby("customer_id")["total_spending"]
    .pct_change()
)

monthly["transaction_change"] = (
    monthly
    .groupby("customer_id")["transaction_count"]
    .pct_change()
)

monthly[
    [
        "customer_id",
        "month",
        "total_spending",
        "spending_change",
        "transaction_change"
    ]
].head(15)

,customer_id,month,total_spending,spending_change,transaction_change
0,100001,2025-03,13351.43,NaN,NaN
1,100001,2025-05,18298.47,0.370525,3.000000
2,100001,2025-06,3416.93,-0.813267,-0.250000
3,100001,2025-07,9934.52,1.907440,-0.333333
4,100001,2025-08,10318.99,0.038700,0.500000
5,100001,2025-09,462.29,-0.955200,-0.666667
6,100001,2025-10,3175.02,5.868027,1.000000
7,100001,2025-11,4212.17,0.326659,0.000000
8,100001,2025-12,6476.41,0.537547,1.000000
9,100002,2025-02,554.81,NaN,NaN


In [8]:
monthly["rolling_3m_spending"] = (
    monthly
    .groupby("customer_id")["total_spending"]
    .transform(
        lambda x: x.rolling(3, min_periods=1).mean()
    )
)

monthly["rolling_3m_volatility"] = (
    monthly
    .groupby("customer_id")["total_spending"]
    .transform(
        lambda x: x.rolling(3, min_periods=2).std()
    )
)

monthly["rolling_3m_volatility"] = (
    monthly["rolling_3m_volatility"]
    .fillna(0)
)

In [9]:
monthly["spending_deviation"] = (
    monthly["total_spending"]
    - monthly["rolling_3m_spending"]
)

monthly["high_spending_growth"] = (
    monthly["spending_change"] > 0.30
).astype(int)

monthly["high_transaction_growth"] = (
    monthly["transaction_change"] > 0.30
).astype(int)

monthly["fraud_flag"] = (
    monthly["fraud_rate"] > 0
).astype(int)

In [10]:
monthly["temporal_stress_score"] = (
    0.30 * monthly["high_spending_growth"]
    + 0.20 * monthly["high_transaction_growth"]
    + 0.25 * monthly["fraud_flag"]
    + 0.25 * (
        monthly["spending_deviation"]
        > 0
    ).astype(int)
)

monthly["temporal_stress_score"] *= 100

monthly[
    [
        "customer_id",
        "month",
        "temporal_stress_score"
    ]
].head(15)

,customer_id,month,temporal_stress_score
0,100001,2025-03,0.0
1,100001,2025-05,75.0
2,100001,2025-06,0.0
3,100001,2025-07,30.0
4,100001,2025-08,45.0
5,100001,2025-09,0.0
6,100001,2025-10,50.0
7,100001,2025-11,55.0
8,100001,2025-12,75.0
9,100002,2025-02,0.0


In [11]:
temporal_profile = (
    monthly
    .groupby("customer_id")
    .agg(
        avg_monthly_spending=(
            "total_spending",
            "mean"
        ),
        spending_volatility_temporal=(
            "total_spending",
            "std"
        ),
        avg_transaction_count=(
            "transaction_count",
            "mean"
        ),
        max_monthly_spending=(
            "total_spending",
            "max"
        ),
        avg_spending_change=(
            "spending_change",
            "mean"
        ),
        max_spending_change=(
            "spending_change",
            "max"
        ),
        avg_transaction_change=(
            "transaction_change",
            "mean"
        ),
        fraud_months=(
            "fraud_flag",
            "sum"
        ),
        avg_temporal_stress=(
            "temporal_stress_score",
            "mean"
        ),
        max_temporal_stress=(
            "temporal_stress_score",
            "max"
        )
    )
    .reset_index()
)

temporal_profile.head()

,customer_id,avg_monthly_spending,spending_volatility_temporal,avg_transaction_count,max_monthly_spending,avg_spending_change,max_spending_change,avg_transaction_change,fraud_months,avg_temporal_stress,max_temporal_stress
0,100001,7738.470000,5711.521310,2.444444,18298.47,0.910054,5.868027,0.531250,0,36.666667,75.0
1,100002,5350.176000,5491.407920,2.200000,15923.88,2.136874,10.508404,0.148148,1,30.500000,80.0
2,100003,3934.038571,4856.459016,1.428571,14549.10,1.946814,10.011118,0.333333,1,35.714286,75.0
3,100004,7872.302000,6264.956803,2.300000,18144.92,1.132483,5.817426,0.355556,2,37.500000,75.0
4,100005,3444.236250,2236.712688,2.000000,7237.67,0.458777,4.863272,0.059524,0,21.875000,75.0


In [12]:
enhanced_profile = customers.merge(
    temporal_profile,
    on="customer_id",
    how="left"
)

enhanced_profile = enhanced_profile.fillna(0)

print(enhanced_profile.shape)

enhanced_profile.head()

(10000, 23)


,customer_id,age,annual_income,employment_years,dependents,savings,existing_debt,credit_score,employment_type,debt_to_income,...,avg_monthly_spending,spending_volatility_temporal,avg_transaction_count,max_monthly_spending,avg_spending_change,max_spending_change,avg_transaction_change,fraud_months,avg_temporal_stress,max_temporal_stress
0,100001,59,6.542970e+05,0,3,281262.935425,185332.880592,708,Contract,0.283255,...,7738.470000,5711.521310,2.444444,18298.47,0.910054,5.868027,0.531250,0,36.666667,75.0
1,100002,49,6.216075e+05,4,3,189110.449367,116700.181303,810,Salaried,0.187739,...,5350.176000,5491.407920,2.200000,15923.88,2.136874,10.508404,0.148148,1,30.500000,80.0
2,100003,35,6.371762e+05,10,2,396507.236148,67320.829790,595,Salaried,0.105655,...,3934.038571,4856.459016,1.428571,14549.10,1.946814,10.011118,0.333333,1,35.714286,75.0
3,100004,63,1.915457e+06,19,4,350710.898806,217865.937714,761,Salaried,0.113741,...,7872.302000,6264.956803,2.300000,18144.92,1.132483,5.817426,0.355556,2,37.500000,75.0
4,100005,28,3.833640e+05,10,4,324423.445016,446028.336260,689,Business Owner,1.163459,...,3444.236250,2236.712688,2.000000,7237.67,0.458777,4.863272,0.059524,0,21.875000,75.0


In [13]:
enhanced_profile.to_csv(
    "C:/Users/lenovo/Desktop/FinRisk-X/data/processed/enhanced_customer_profile.csv",
    index=False
)

monthly.to_csv(
    "C:/Users/lenovo/Desktop/FinRisk-X/data/processed/monthly_transaction_behavior.csv",
    index=False
)

print("Temporal datasets saved successfully.")

Temporal datasets saved successfully.


In [14]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report
)

from xgboost import XGBClassifier

enhanced_profile = pd.read_csv(
    "../data/processed/enhanced_customer_profile.csv"
)

print(enhanced_profile.shape)
enhanced_profile.head()

(10000, 23)


,customer_id,age,annual_income,employment_years,dependents,savings,existing_debt,credit_score,employment_type,debt_to_income,...,avg_monthly_spending,spending_volatility_temporal,avg_transaction_count,max_monthly_spending,avg_spending_change,max_spending_change,avg_transaction_change,fraud_months,avg_temporal_stress,max_temporal_stress
0,100001,59,6.542970e+05,0,3,281262.935425,185332.880592,708,Contract,0.283255,...,7738.470000,5711.521310,2.444444,18298.47,0.910054,5.868027,0.531250,0,36.666667,75.0
1,100002,49,6.216075e+05,4,3,189110.449367,116700.181303,810,Salaried,0.187739,...,5350.176000,5491.407920,2.200000,15923.88,2.136874,10.508404,0.148148,1,30.500000,80.0
2,100003,35,6.371762e+05,10,2,396507.236148,67320.829790,595,Salaried,0.105655,...,3934.038571,4856.459016,1.428571,14549.10,1.946814,10.011118,0.333333,1,35.714286,75.0
3,100004,63,1.915457e+06,19,4,350710.898806,217865.937714,761,Salaried,0.113741,...,7872.302000,6264.956803,2.300000,18144.92,1.132483,5.817426,0.355556,2,37.500000,75.0
4,100005,28,3.833640e+05,10,4,324423.445016,446028.336260,689,Business Owner,1.163459,...,3444.236250,2236.712688,2.000000,7237.67,0.458777,4.863272,0.059524,0,21.875000,75.0


In [15]:
temporal_features = [
    "avg_monthly_spending",
    "spending_volatility_temporal",
    "avg_transaction_count",
    "max_monthly_spending",
    "avg_spending_change",
    "max_spending_change",
    "avg_transaction_change",
    "fraud_months",
    "avg_temporal_stress",
    "max_temporal_stress"
]

missing = [
    col for col in temporal_features
    if col not in enhanced_profile.columns
]

if missing:
    raise ValueError(f"Missing temporal features: {missing}")

print("All temporal features are available.")

All temporal features are available.


In [16]:
X = enhanced_profile[temporal_features].copy()
y = enhanced_profile["default"].copy()

X = X.replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

print("X shape:", X.shape)
print("Target distribution:")
print(y.value_counts())

X shape: (10000, 10)
Target distribution:
default
0    6781
1    3219
Name: count, dtype: int64


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 8000
Testing samples: 2000


In [18]:
temporal_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

temporal_model.fit(
    X_train,
    y_train
)

print("Temporal XGBoost trained successfully.")

Temporal XGBoost trained successfully.


In [19]:
y_pred = temporal_model.predict(X_test)
y_prob = temporal_model.predict_proba(X_test)[:, 1]

temporal_results = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(
        y_test,
        y_pred,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test,
        y_pred,
        zero_division=0
    ),
    "F1": f1_score(
        y_test,
        y_pred,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_test,
        y_prob
    ),
    "PR-AUC": average_precision_score(
        y_test,
        y_prob
    )
}

temporal_results

{'Accuracy': 0.676,
 'Precision': 0.4166666666666667,
 'Recall': 0.015527950310559006,
 'F1': 0.029940119760479042,
 'ROC-AUC': 0.49440260906209343,
 'PR-AUC': 0.3285940395827721}

In [20]:
baseline_model = joblib.load(
    "../models/credit_risk_xgb.pkl"
)

baseline_features = joblib.load(
    "../models/credit_risk_features.pkl"
)

print("Baseline features:", len(baseline_features))

Baseline features: 25


In [21]:
baseline_X = enhanced_profile[
    baseline_features
].copy()

baseline_X = baseline_X.replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

_, baseline_test, _, baseline_y_test = train_test_split(
    baseline_X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

baseline_prob = baseline_model.predict_proba(
    baseline_test
)[:, 1]

baseline_auc = roc_auc_score(
    baseline_y_test,
    baseline_prob
)

baseline_pr_auc = average_precision_score(
    baseline_y_test,
    baseline_prob
)

print(f"Baseline ROC-AUC: {baseline_auc:.4f}")
print(f"Baseline PR-AUC:  {baseline_pr_auc:.4f}")

KeyError: "['total_loans', 'total_loan_amount', 'average_interest_rate', 'average_loan_term', 'total_monthly_payment', 'total_transactions', 'total_spending', 'average_transaction', 'transaction_std', 'maximum_transaction', 'fraudulent_transactions', 'transaction_frequency', 'spending_volatility', 'fraud_ratio', 'loan_to_income', 'payment_to_income', 'savings_to_loan'] not in index"

In [24]:
# Load the original static financial profile
static_profile = pd.read_csv(
    "C:/Users/lenovo/Desktop/FinRisk-X/data/processed/customer_financial_profile.csv"
)

print("Static profile shape:", static_profile.shape)
print("Enhanced profile shape:", enhanced_profile.shape)

# Merge static + temporal features
full_profile = static_profile.merge(
    enhanced_profile[
        ["customer_id"] + temporal_features
    ],
    on="customer_id",
    how="inner"
)

print("Combined profile shape:", full_profile.shape)

Static profile shape: (10000, 33)
Enhanced profile shape: (10000, 23)
Combined profile shape: (10000, 43)


In [25]:
missing = [
    col for col in baseline_features
    if col not in full_profile.columns
]

print("Missing baseline features:", missing)

Missing baseline features: []


In [26]:
baseline_X = full_profile[baseline_features].copy()
baseline_y = full_profile["default"].copy()

baseline_X = baseline_X.replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

_, baseline_test, _, baseline_y_test = train_test_split(
    baseline_X,
    baseline_y,
    test_size=0.20,
    stratify=baseline_y,
    random_state=42
)

baseline_prob = baseline_model.predict_proba(
    baseline_test
)[:, 1]

baseline_auc = roc_auc_score(
    baseline_y_test,
    baseline_prob
)

baseline_pr_auc = average_precision_score(
    baseline_y_test,
    baseline_prob
)

print(f"Baseline ROC-AUC: {baseline_auc:.4f}")
print(f"Baseline PR-AUC:  {baseline_pr_auc:.4f}")

Baseline ROC-AUC: 0.6727
Baseline PR-AUC:  0.4900
